In [6]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash
JupyterDash.infer_jupyter_proxy_config()

# Dashboard components
import dash_leaflet as dl
from dash import dcc, html, dash_table
from dash.dependencies import Input, Output
import plotly.express as px

# Utilities
import base64
import pandas as pd

#### FIX ME (already done for you) ####
# Change this import to match YOUR CRUD Python module file/class:
# If your file is AnimalShelter.py and class AnimalShelter, use:
from AnimalShelter import AnimalShelter


###########################
# Data Manipulation / Model
###########################
username = "aacuser"
password = "SNHU1234"

# Connect to database via CRUD Module
db = AnimalShelter(username, password)

# Pull all records for initial load
df = pd.DataFrame.from_records(db.read({}))

# Drop MongoDB ObjectId to prevent DataTable crash
if '_id' in df.columns:
    df.drop(columns=['_id'], inplace=True)

# Safety: If location columns are missing, the map will fail.
# (Normally they exist in CS340 dataset)
# print(df.columns)


#########################
# Dashboard Layout / View
#########################
app = JupyterDash(__name__)

# Grazioso Salvare logo (base64 embedded)
# IMPORTANT: Update filename to the actual logo file you have
image_filename = 'Grazioso Salvare Logo.png'  # <-- change if needed (example: 'Grazioso_Salvare_Logo.png')
encoded_image = base64.b64encode(open(image_filename, 'rb').read()).decode('utf-8')

app.layout = html.Div([

    html.Div([
        html.Div([
            html.Img(
                src=f"data:image/png;base64,{encoded_image}",
                style={'height': '90px', 'marginRight': '15px'}
            ),
        ], style={'display': 'flex', 'justifyContent': 'center'}),

        html.H1("Grazioso Salvare Rescue Dashboard", style={'marginBottom': '0px'}),
        html.H4("Ryan Eames | CS 340 Project Two", style={'marginTop': '5px'})  # Unique Identifier
    ], style={'textAlign': 'center'}),

    html.Hr(),

    # Interactive filtering options (Radio Items)
    html.Div([
        html.Label("Rescue Filter:", style={'fontWeight': 'bold'}),
        dcc.RadioItems(
            id='filter-type',
            options=[
                {'label': 'Reset (All)', 'value': 'RESET'},
                {'label': 'Water Rescue', 'value': 'WATER'},
                {'label': 'Mountain/Wilderness Rescue', 'value': 'MOUNTAIN'},
                {'label': 'Disaster/Individual Tracking', 'value': 'DISASTER'}
            ],
            value='RESET',
            inline=True
        )
    ], style={'padding': '10px 20px'}),

    html.Hr(),

    # DataTable
    dash_table.DataTable(
        id='datatable-id',
        columns=[{"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns],
        data=df.to_dict('records'),

        # User-friendly features
        page_size=10,
        page_action="native",
        sort_action="native",
        filter_action="native",
        row_selectable="single",
        selected_rows=[0],

        style_table={'overflowX': 'auto'},
        style_cell={'textAlign': 'left', 'padding': '6px', 'fontFamily': 'Arial', 'fontSize': '14px'},
        style_header={'fontWeight': 'bold'}
    ),

    html.Br(),
    html.Hr(),

    # Side-by-side chart + map
    html.Div(
        className='row',
        style={'display': 'flex', 'gap': '10px'},
        children=[
            html.Div(id='graph-id', className='col s12 m6', style={'flex': '1'}),
            html.Div(id='map-id', className='col s12 m6', style={'flex': '1'})
        ]
    )
])


#############################################
# Interaction Between Components / Controller
#############################################

def build_query(filter_type: str) -> dict:
    """
    Build Mongo queries for each rescue filter option.
    NOTE: If your Dashboard Specs Document has different breed lists,
    swap the breed arrays below to match it.
    """
    if filter_type == "WATER":
        return {
            "animal_type": "Dog",
            "breed": {"$in": ["Labrador Retriever Mix", "Chesapeake Bay Retriever", "Newfoundland"]},
            "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156}
        }

    if filter_type == "MOUNTAIN":
        return {
            "animal_type": "Dog",
            "breed": {"$in": ["German Shepherd", "Alaskan Malamute", "Old English Sheepdog", "Siberian Husky", "Rottweiler"]},
            "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156}
        }

    if filter_type == "DISASTER":
        return {
            "animal_type": "Dog",
            "breed": {"$in": ["Doberman Pinscher", "German Shepherd", "Golden Retriever", "Bloodhound", "Rottweiler"]},
            "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156}
        }

    # RESET / default
    return {}


@app.callback(
    Output('datatable-id', 'data'),
    Input('filter-type', 'value')
)
def update_dashboard(filter_type):
    query = build_query(filter_type)
    dff = pd.DataFrame.from_records(db.read(query))
    if '_id' in dff.columns:
        dff.drop(columns=['_id'], inplace=True)
    return dff.to_dict('records')


@app.callback(
    Output('graph-id', 'children'),
    Input('datatable-id', 'derived_virtual_data')
)
def update_graphs(viewData):
    if viewData is None or len(viewData) == 0:
        return html.Div("No data available for chart.")

    dff = pd.DataFrame.from_dict(viewData)

    # Chart choice: Top 10 breeds bar chart
    if 'breed' not in dff.columns:
        return html.Div("Missing 'breed' column for chart.")

    breed_counts = dff['breed'].value_counts().head(10).reset_index()
    breed_counts.columns = ['breed', 'count']

    fig = px.bar(
        breed_counts,
        x='breed',
        y='count',
        title='Top Breeds (Current Filter)',
        labels={'breed': 'Breed', 'count': 'Count'}
    )

    return dcc.Graph(figure=fig)


@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    Input('datatable-id', 'selected_columns')
)
def update_styles(selected_columns):
    if not selected_columns:
        return []
    return [{
        'if': {'column_id': i},
        'backgroundColor': '#D2F3FF'
    } for i in selected_columns]


@app.callback(
    Output('map-id', 'children'),
    [Input('datatable-id', 'derived_virtual_data'),
     Input('datatable-id', 'derived_virtual_selected_rows')]
)
def update_map(viewData, selected_rows):
    if viewData is None or len(viewData) == 0:
        return html.Div("No data available for map.")

    dff = pd.DataFrame.from_dict(viewData)

    # Pick first row if nothing is selected
    row = 0
    if selected_rows and len(selected_rows) > 0:
        row = selected_rows[0]

    # Required columns for mapping
    required_cols = ['location_lat', 'location_long']
    for c in required_cols:
        if c not in dff.columns:
            return html.Div(f"Missing '{c}' column for map.")

    lat = dff.loc[row, 'location_lat']
    lon = dff.loc[row, 'location_long']

    # Optional hover fields (won’t crash if missing)
    breed = dff.loc[row, 'breed'] if 'breed' in dff.columns else "Unknown breed"
    name = dff.loc[row, 'name'] if 'name' in dff.columns else "Unknown name"

    # Austin TX center
    return [
        dl.Map(
            style={'width': '100%', 'height': '500px'},
            center=[30.75, -97.48],
            zoom=10,
            children=[
                dl.TileLayer(id="base-layer-id"),
                dl.Marker(
                    position=[lat, lon],
                    children=[
                        dl.Tooltip(str(breed)),
                        dl.Popup([
                            html.H3("Animal Details"),
                            html.P(f"Name: {name}"),
                            html.P(f"Breed: {breed}")
                        ])
                    ]
                )
            ]
        )
    ]


# Run app
app.run_server(debug=True)

Dash app running on https://rondoshine-alpinesnake-3000.codio.io/proxy/8050/
